# Next-month spend

Classification asks whether a customer places no paid order in July. Regression asks how many baht they spend that month.

`next_month_spend_thb` is 0 when `churned` is 1 and positive otherwise. Neither column is a feature.

Use the same split as the churn notebooks. Compare a mean baseline with a small ridge model on validation MAE, then score test rows once. Do not use ROC-AUC here.

Follow the cells below for the timed comparison. A larger search is optional.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "schema.py").is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "schema.py").is_file() and (nested / "app.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder, from 01-supervised-learning, or from its notebooks folder."
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from pipelines import make_dummy_spend_pipeline, tune_ridge
from schema import SPEND_TARGET
from spend import spend_scores

y_train_spend = train[SPEND_TARGET]
y_val_spend = validation[SPEND_TARGET]
y_test_spend = test[SPEND_TARGET]
dummy = make_dummy_spend_pipeline()
dummy.fit(X_train, y_train_spend)
search = tune_ridge(X_train, y_train_spend, n_iter=4)
ridge = search.best_estimator_
print(search.best_params_)


### Exercise

`spend_scores` returns `mae_thb`, `rmse_thb`, and `r2` after clipping predictions at 0 THB. A negative prediction is not a possible spend.

Assign the validation MAE for the dummy model and for the tuned ridge model.


In [ ]:
dummy_val_mae = None
ridge_val_mae = None


In [ ]:
if dummy_val_mae is None or ridge_val_mae is None:
    raise NotImplementedError("Finish the validation MAE exercise.")
print("dummy validation MAE", round(float(dummy_val_mae), 2))
print("ridge validation MAE", round(float(ridge_val_mae), 2))
winner = ridge if ridge_val_mae <= dummy_val_mae else dummy
print("save", "ridge" if winner is ridge else "dummy")


### Exercise. Test set, once

Assign test MAE and test RMSE in THB for `winner`. Use `spend_scores` on the test rows. Run this once.


In [ ]:
test_mae = None
test_rmse = None


In [ ]:
if test_mae is None or test_rmse is None:
    raise NotImplementedError("Assign test MAE and test RMSE. Run the test cell once.")
print("test MAE", round(float(test_mae), 2))
print("test RMSE", round(float(test_rmse), 2))


In [ ]:
from persist import save_spend_bundle

destination = ROOT / "models" / "spend_bundle.joblib"
save_spend_bundle(destination, winner)
print("saved", destination)
print("Load only this file or one your instructor hands you.")
print("A joblib file can run code when it is loaded.")


## Checkpoint

The form reads both bundles. A churn file in the spend slot is rejected. Next, open `app.py`.
